# Official MJD Transformer Training

Runs the standardized 3-tokenization × 2-position-encoding matrix for both MJD tasks. Data splitting, preprocessing, optimization, checkpointing, and evaluation come from the unchanged collaborator-provided `mjdbench` workflow.

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import pandas as pd
import torch

configured_root = os.environ.get('MJD_TRANSFORMER_PROJECT_ROOT')
candidate_roots = []
if configured_root:
    candidate_roots.append(Path(configured_root).expanduser())
candidate_roots.extend([
    Path.cwd() / 'mjd_detector',
    Path.cwd(),
    Path.cwd().parent / 'mjd_detector',
    Path.cwd().parent,
    Path.cwd().parent.parent / 'mjd_detector',
])
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in candidate_roots
     if (candidate / 'mjd_transformer').is_dir()
     and (candidate / 'mjdbench').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate mjd_detector; set MJD_TRANSFORMER_PROJECT_ROOT.'
    )
project_string = str(PROJECT_ROOT)
if project_string not in sys.path:
    sys.path.insert(0, project_string)

from mjdbench import (
    DataConfig,
    TrainingConfig,
    evaluate_model,
    prepare_dataset,
    set_seed,
    train_model,
)

from mjd_transformer import MJDTransformer, TokenizationConfig

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Transformer package:', Path(sys.modules['mjd_transformer'].__file__).resolve())

In [ ]:
DATA_ROOT = Path(os.environ.get(
    'MJD_BENCH_DATA',
    str(PROJECT_ROOT.parent / 'data' / 'MJD'),
)).expanduser()
OUTPUT_ROOT = Path(os.environ.get(
    'MJD_OUTPUT_ROOT',
    str(PROJECT_ROOT / 'results' / 'transformer_official_v1'),
)).expanduser()
SUMMARY_PATH = OUTPUT_ROOT / 'transformer_results.csv'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

data_config = DataConfig(
    data_root=DATA_ROOT,
    validation_fraction=0.10,
    baseline_samples=200,
    classification_amplitude_normalization=True,
    regression_waveform_scale=1.0,
    seed=42,
)
training_config = TrainingConfig()  # Shared collaborator defaults.
TASKS = ('classification', 'regression')

# Add a run ID here only when you intentionally want to skip it manually.
COMPLETED_OFFICIAL_RUNS = set()

print('Dataset:', DATA_ROOT)
print('Outputs:', OUTPUT_ROOT)
print('Data config:', data_config.to_dict())
print('Training config:', training_config.to_dict())

In [ ]:
EXPERIMENTS = [
    {'tokenization': tokenization, 'position_encoding': position_encoding}
    for tokenization in ('raw_patches', 'segment_summary', 'pulse_entities')
    for position_encoding in ('coordinate_mlp', 'fourier_coordinates')
]
ALL_RUN_IDS = {
    f'{task}__{experiment["tokenization"]}__{experiment["position_encoding"]}'
    for task in TASKS
    for experiment in EXPERIMENTS
}
requested_runs = os.environ.get('MJD_RUN_IDS', '').strip()
SELECTED_RUN_IDS = (
    {value.strip() for value in requested_runs.split(',') if value.strip()}
    if requested_runs else set()
)
unknown_runs = SELECTED_RUN_IDS - ALL_RUN_IDS
if unknown_runs:
    raise ValueError(f'Unknown MJD_RUN_IDS: {sorted(unknown_runs)}')

def make_tokenization_config(name):
    if name == 'raw_patches':
        return TokenizationConfig(
            tokenization='raw_patches',
            patch_size=20,
        )
    if name == 'segment_summary':
        return TokenizationConfig(
            tokenization='segment_summary',
            token_count=500,
        )
    if name == 'pulse_entities':
        return TokenizationConfig(
            tokenization='pulse_entities',
            token_count=500,
            uniform_entity_fraction=0.5,
            entity_context_size=9,
        )
    raise ValueError(f'Unknown tokenization: {name}')

def run_is_complete(run_dir):
    required = ('run_config.json', 'best.pt', 'history.json', 'metrics.json', 'run_summary.json')
    return all((run_dir / name).is_file() for name in required)

def upsert_summary(row):
    if SUMMARY_PATH.is_file():
        table = pd.read_csv(SUMMARY_PATH)
        table = table.loc[table['run_id'] != row['run_id']]
    else:
        table = pd.DataFrame()
    table = pd.concat([table, pd.DataFrame([row])], ignore_index=True)
    table = table.sort_values(['task', 'tokenization', 'position_encoding'])
    table.to_csv(SUMMARY_PATH, index=False)

experiment_table = pd.DataFrame([
    {
        'run_id': f'{task}__{experiment["tokenization"]}__{experiment["position_encoding"]}',
        'task': task,
        **experiment,
    }
    for task in TASKS
    for experiment in EXPERIMENTS
])
display(experiment_table)
print('Selected runs:', sorted(SELECTED_RUN_IDS) if SELECTED_RUN_IDS else 'all')

In [ ]:
for task in TASKS:
    if SELECTED_RUN_IDS and not any(
        run_id.startswith(f'{task}__') for run_id in SELECTED_RUN_IDS
    ):
        continue
    print('\n' + '#' * 88)
    print(f'Preparing official {task} data once')
    print('#' * 88)
    task_data = prepare_dataset(
        task=task,
        data_config=data_config,
        batch_size=training_config.batch_size,
        num_workers=training_config.num_workers,
        limit_per_split=None,
    )
    print('Counts:', task_data.counts)

    for experiment in EXPERIMENTS:
        tokenization = experiment['tokenization']
        position_encoding = experiment['position_encoding']
        run_id = f'{task}__{tokenization}__{position_encoding}'
        run_dir = OUTPUT_ROOT / run_id

        if SELECTED_RUN_IDS and run_id not in SELECTED_RUN_IDS:
            continue

        if run_id in COMPLETED_OFFICIAL_RUNS or run_is_complete(run_dir):
            print(f'Skipping completed run: {run_id}')
            continue

        print('\n' + '=' * 88)
        print(run_id)
        print('=' * 88)
        run_dir.mkdir(parents=True, exist_ok=True)
        tokenization_config = make_tokenization_config(tokenization)

        # Seed before construction so initial weights are reproducible.
        set_seed(training_config.seed, training_config.deterministic)
        model = MJDTransformer(
            task=task,
            tokenization_config=tokenization_config,
            position_encoding=position_encoding,
            d_model=64,
            nhead=4,
            num_layers=2,
            dim_feedforward=256,
            dropout=0.1,
            num_frequencies=6,
        )
        parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
        run_config = {
            'run_id': run_id,
            'task': task,
            'data': data_config.to_dict(),
            'counts': task_data.counts,
            'training': training_config.to_dict(),
            'representation': model.config_dict(),
            'parameter_count': parameter_count,
        }
        (run_dir / 'run_config.json').write_text(
            json.dumps(run_config, indent=2), encoding='utf-8'
        )
        print('Trainable parameters:', f'{parameter_count:,}')

        training_start = time.perf_counter()
        history = train_model(
            model,
            task_data.train_loader,
            task_data.validation_loader,
            task=task,
            config=training_config,
            output_dir=run_dir,
        )
        training_seconds = time.perf_counter() - training_start

        evaluation_start = time.perf_counter()
        metrics = evaluate_model(
            model,
            task_data.test_loader,
            task=task,
            device=training_config.device,
            output_dir=run_dir,
            use_amp=training_config.use_amp,
            amp_precision=training_config.amp_precision,
        )
        evaluation_seconds = time.perf_counter() - evaluation_start
        checkpoint = torch.load(run_dir / 'best.pt', map_location='cpu', weights_only=False)
        epochs_completed = len(history)
        row = {
            'run_id': run_id,
            'task': task,
            'tokenization': tokenization,
            'position_encoding': position_encoding,
            'parameter_count': parameter_count,
            'train_events': task_data.counts['train'],
            'validation_events': task_data.counts['validation'],
            'test_events': task_data.counts['test'],
            'epochs_completed': epochs_completed,
            'best_epoch': int(checkpoint['epoch']),
            'best_validation_score': float(checkpoint['score']),
            'training_seconds': training_seconds,
            'minutes_per_epoch': training_seconds / max(epochs_completed, 1) / 60.0,
            'evaluation_seconds': evaluation_seconds,
            'test_auc': metrics.get('auc'),
            'test_accuracy': metrics.get('accuracy'),
            'test_mae_kev': metrics.get('mae_kev'),
            'test_rmse_kev': metrics.get('rmse_kev'),
            'test_bias_kev': metrics.get('bias_kev'),
            'test_loss': metrics.get('loss'),
        }
        (run_dir / 'run_summary.json').write_text(
            json.dumps(row, indent=2, allow_nan=True), encoding='utf-8'
        )
        upsert_summary(row)
        print(json.dumps(row, indent=2, allow_nan=True))

In [ ]:
results = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.is_file() else pd.DataFrame()
print('Completed official runs:', len(results))
display(results)